# W04_4 위성영상으로 지붕 읽기 — 기복변위 보정과 옥상 판독

<a href="https://colab.research.google.com/github/kloud80/dataminig_cau/blob/main/%EC%A4%91%EC%95%99%EB%8C%8026%ED%8A%B9%EA%B0%95/notebooks/w04_colab/W04_4%20%EC%9C%84%EC%84%B1%EC%98%81%EC%83%81%EC%9C%BC%EB%A1%9C%20%EC%A7%80%EB%B6%95%20%EC%9D%BD%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

4주차 주제는 **주소 한 줄에서 3D 건물까지**다. 앞 노트북에서 주소 → 좌표 → 건물 윤곽·높이(공공데이터)까지 왔다.
이번에는 그 윤곽 **위**, 지붕을 위성영상으로 읽는다.

연구실에서 쓰는 파이프라인 `addr2blend` 의 지붕 부분을 단순하게 다시 만든다.

| 단계 | 이 노트북 | addr2blend 원본 | 하는 일 |
|---|---|---|---|
| 1 | 정사영상 받기 | `s03_prior` · `sources/ortho.py` `local_ortho` | VWorld 위성 타일 → 건물 중심 로컬 좌표(m) 영상 |
| 2 | 기복변위 찾기 | `estimate_roof_shift` | 외곽선을 에지맵 위에서 밀어 보며 지붕이 밀린 양을 찾는다 |
| 3 | 지붕 잘라내기 | `aligned_roof_crop` | 밀린 만큼 옮기고, 건물 주축으로 세워 자른다 |
| 4 | 옥상 판독 | `bds/roof_reader.py` | 비전 모델이 계단실·실외기 위치를 **JSON 스키마**로 답한다 |
| 5 | 넘기기 | → BDS `roof.items` | `out/roof_items.json` → 5번 노트북이 3D 로 세운다 |

**예제 건물** 아이파크타워 (서울 강남구 영동대로 520) — 지상 15층, 높이 94.1 m (건축물대장)

> **키가 없어도 끝까지 돈다.** `VWORLD_KEY` 가 없으면 저장해 둔 정사영상을, `OPENAI_API_KEY` 가 없으면
> 실제 파이프라인이 낸 판독 결과를 쓴다. 키는 Colab 왼쪽 열쇠 아이콘(보안 비밀)에 넣는다 — 노트북에 직접 적지 말 것.

## 1. 준비

In [ ]:
import os, sys, io, json, math, time, base64, subprocess, importlib
from pathlib import Path

# ── 데이터 찾기 ─────────────────────────────────────────────
REPO_URL = "https://github.com/kloud80/dataminig_cau.git"  # 강의 저장소
REPO_DIR = "중앙대26특강/notebooks/w04_colab"   # 저장소 안에서 이 노트북들이 있는 폴더

def find_base():
    for b in [Path.cwd(), Path.cwd() / REPO_DIR, Path("/content") / REPO_DIR, Path("/content/repo") / REPO_DIR]:
        if (b / "data" / "ipark" / "s04_geometry.json").exists():
            return b
    return None

BASE = find_base()
if BASE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL, "/content/repo"], check=False)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", REPO_DIR], check=False)
    BASE = find_base()
if BASE is None:
    raise SystemExit(
        "data/ipark 폴더를 찾지 못했습니다.\n"
        "  (1) 네트워크를 확인하고 이 셀을 다시 실행 (강의 저장소에서 자동으로 받는다)\n"
        "  (2) 또는 왼쪽 파일 탭에 w04_colab 폴더(data/ipark 포함)를 통째로 올린 뒤 다시 실행")
DATA = BASE / "data" / "ipark"
OUT = BASE / "out"
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(BASE))          # w04_tools.py (보기용 도우미)

# ── 패키지 (Colab 에 없는 것만 설치) ────────────────────────
def need(mod, pip_name=None):
    try:
        return importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=True)
        return importlib.import_module(mod)

import numpy as np
import cv2
import requests
from PIL import Image
need("pyproj")
from pyproj import Transformer

# ── API 키: Colab 보안 비밀 → 환경변수. 값은 절대 출력하지 않는다 ──
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(name, "").strip()
    return v or None

VWORLD_KEY = get_key("VWORLD_KEY")
OPENAI_KEY = get_key("OPENAI_API_KEY")
print("VWORLD_KEY    :", "있음" if VWORLD_KEY else "없음 → 저장된 정사영상 사용")
print("OPENAI_API_KEY:", "있음" if OPENAI_KEY else "없음 → 실제 파이프라인의 판독 결과 사용")
print("데이터:", DATA)
print("결과물:", OUT)

In [ ]:
# 한글 폰트 — 그래프 글자가 네모로 깨지지 않게 (Colab 은 나눔고딕을 설치, 윈도는 맑은 고딕)
import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
FONT_PATH = None
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgunbd.ttf'
else:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf'
    if not os.path.exists(nanum):
        os.system('apt-get -qq install -y fonts-nanum > /dev/null')
    if os.path.exists(nanum):
        fm.fontManager.addfont(nanum)
        fm.fontManager.addfont(nanum.replace('Bold', ''))
        plt.rcParams['font.family'] = 'NanumGothic'
        FONT_PATH = nanum
plt.rcParams['axes.unicode_minus'] = False

## 2. 이미 아는 것 — 공공데이터가 준 윤곽과 높이

파이프라인은 건물 중심을 원점으로 하는 **로컬 좌표계 `LOCAL_ENU`** 에서 일한다.
+X = 동, +Y = 북, 단위는 미터. 경위도를 그대로 쓰면 "1 m 옮겨라" 같은 계산이 안 되기 때문이다.

* 윤곽(footprint) — 국가 GIS 건물 외곽선
* 높이·층수 — 건축물대장
* 입면(facade) — 윤곽의 변 하나하나. `f_0` 이 영동대로를 보는 70 m 짜리 긴 면이다.

In [ ]:
prior = json.loads((DATA / "s03_prior.json").read_text(encoding="utf-8"))
geom = json.loads((DATA / "s04_geometry.json").read_text(encoding="utf-8"))

FP = np.array(geom["footprint"])                 # [[x, y], ...] m
H = float(geom["height_m"])
LNG0, LAT0 = geom["origin_lnglat"]
name = (prior.get("juso") or {}).get("bd_nm") or ""
print(f"{name} — 높이 {H:.1f} m ({geom['height_source']}), 지상 {len(geom['floors'])}층, "
      f"사용승인 {prior.get('approval_date')}")
print(f"원점 경위도 {LNG0:.6f}, {LAT0:.6f} · 외곽선 꼭짓점 {len(FP)}개 · 입면 {len(geom['facades'])}개")

fig, ax = plt.subplots(figsize=(6, 7))
ax.fill(*FP.T, color="#dfe6ee", ec="#33475b", lw=2)
for f in geom["facades"]:
    (sx, sy), (ex, ey) = f["start"], f["end"]
    mx, my = (sx + ex) / 2, (sy + ey) / 2
    nx, ny = f["normal"]
    if f["width_m"] > 8:
        ax.annotate(f'{f["id"]}\n{f["width_m"]:.0f} m', (mx + nx * 7, my + ny * 7), ha="center", va="center", fontsize=10)
ax.plot(0, 0, "k+", ms=12)
ax.annotate("N", (28, 30), (28, 18), arrowprops=dict(arrowstyle="->", lw=2), ha="center", fontsize=13)
ax.set_aspect("equal"); ax.set_xlabel("X 동쪽 (m)"); ax.set_ylabel("Y 북쪽 (m)")
ax.set_title("아이파크타워 외곽선 (LOCAL_ENU)"); ax.grid(alpha=.3); plt.show()

## 3. 정사영상 받기 — VWorld WMTS 위성 타일

VWorld 는 위성영상을 **256×256 타일**로 나눠 준다(Web Mercator). 줌 19 면 한 타일이 서울에서 약 60 m, 픽셀당 약 0.24 m 다.
타일 몇 장을 이어 붙인 뒤, 로컬 좌표의 **0.25 m 격자**로 다시 표본화한다 — 이제 영상의 픽셀이 곧 미터다.

여백은 높이의 60 % (최대 90 m). 높은 건물일수록 지붕이 외곽선 밖으로 멀리 밀려 찍히기 때문이다 (4절).

* `VWORLD_KEY` 가 있으면 실제로 받는다. VWorld 키는 발급할 때 등록한 **서비스 URL** 을 `Referer` 로 보내야 한다.
* 없거나 실패하면 파이프라인이 받아 둔 `ortho_local.png` 를 쓴다 (같은 방식으로 만든 영상).

In [ ]:
VWORLD_DOMAIN = "http://localhost"   # 키를 발급받을 때 등록한 서비스 URL 로 바꿀 것
ZOOM, MPP = 19, 0.25

PROJ = f"+proj=tmerc +lat_0={LAT0:.10f} +lon_0={LNG0:.10f} +k=1 +x_0=0 +y_0=0 +ellps=GRS80 +units=m +no_defs"
TO_LNGLAT = Transformer.from_crs(PROJ, "EPSG:4326", always_xy=True)

def lnglat_to_tile(lng, lat, z):
    """경위도 → Web Mercator 타일 좌표(실수). 정수부가 타일 번호, 소수부가 타일 안 위치."""
    n = 2 ** z
    x = (np.asarray(lng, float) + 180.0) / 360.0 * n
    y = (1.0 - np.arcsinh(np.tan(np.radians(np.asarray(lat, float)))) / np.pi) / 2.0 * n
    return x, y

def fetch_tile(sess, z, x, y):
    url = f"https://api.vworld.kr/req/wmts/1.0.0/{VWORLD_KEY}/Satellite/{z}/{y}/{x}.jpeg"
    r = sess.get(url, headers={"Referer": VWORLD_DOMAIN}, timeout=10)
    if r.status_code != 200 or not r.headers.get("content-type", "").startswith("image"):
        return None
    return np.array(Image.open(io.BytesIO(r.content)).convert("RGB"))

def vworld_ortho(bounds):
    """로컬 bbox (minx, miny, maxx, maxy) → 북쪽이 위인 RGB 영상 (MPP m/px)."""
    minx, miny, maxx, maxy = bounds
    lng, lat = TO_LNGLAT.transform([minx, maxx], [miny, maxy])
    x0f, y1f = lnglat_to_tile(min(lng), min(lat), ZOOM)
    x1f, y0f = lnglat_to_tile(max(lng), max(lat), ZOOM)
    xs, ys = range(int(x0f), int(x1f) + 1), range(int(y0f), int(y1f) + 1)
    if len(xs) * len(ys) > 120:
        raise RuntimeError("타일이 너무 많다")
    sess, got, rows = requests.Session(), 0, []
    for ty in ys:
        row = []
        for tx in xs:
            t = fetch_tile(sess, ZOOM, tx, ty)
            got += t is not None
            row.append(t if t is not None else np.full((256, 256, 3), 128, np.uint8))
        rows.append(np.concatenate(row, axis=1))
    if got < 0.5 * len(xs) * len(ys):
        raise RuntimeError(f"타일을 {got}/{len(xs) * len(ys)} 장만 받았다 (키·서비스 URL 확인)")
    mosaic = np.concatenate(rows, axis=0)
    # 로컬 격자의 각 픽셀이 모자이크의 어디에 해당하는지 계산해 다시 표본화
    W, Hh = int(math.ceil((maxx - minx) / MPP)), int(math.ceil((maxy - miny) / MPP))
    gx, gy = np.meshgrid(minx + (np.arange(W) + .5) * MPP, maxy - (np.arange(Hh) + .5) * MPP)
    lg, lt = TO_LNGLAT.transform(gx.ravel(), gy.ravel())
    tx, ty = lnglat_to_tile(lg, lt, ZOOM)
    mapx = ((tx - xs[0]) * 256).reshape(Hh, W).astype(np.float32)
    mapy = ((ty - ys[0]) * 256).reshape(Hh, W).astype(np.float32)
    return cv2.remap(mosaic, mapx, mapy, cv2.INTER_LINEAR), f"VWorld 위성 타일 {got}장 (z{ZOOM})"

need_m = float(min(90.0, max(10.0, 0.6 * H)))        # 기복변위를 담을 여유
BOUNDS = (*(FP.min(0) - need_m), *(FP.max(0) + need_m))
IMG, SRC = None, ""
if VWORLD_KEY:
    try:
        t0 = time.time()
        IMG, SRC = vworld_ortho(BOUNDS)
        SRC += f", {time.time() - t0:.1f}초"
    except Exception as e:
        print("VWorld 실패 →", type(e).__name__, str(e)[:200])
if IMG is None:
    IMG = np.array(Image.open(DATA / "ortho_local.png").convert("RGB"))
    BOUNDS = tuple(prior["ortho_bounds_local"])
    MPP = float(prior["ortho_m_per_px"])
    SRC = "저장된 정사영상 data/ipark/ortho_local.png"
Image.fromarray(IMG).save(OUT / "ortho_local.png")
print(f"출처: {SRC}\n영상 {IMG.shape[1]}×{IMG.shape[0]} px = {IMG.shape[1] * MPP:.0f} × {IMG.shape[0] * MPP:.0f} m, 여백 {need_m:.0f} m")

def to_px(xy, bounds=None, mpp=None):
    """로컬 좌표(m) → 영상 픽셀 (열, 행). 영상은 북쪽이 위라 y 부호가 뒤집힌다."""
    minx, miny, maxx, maxy = bounds or BOUNDS
    m = mpp or MPP
    xy = np.asarray(xy, float)
    return np.column_stack([(xy[:, 0] - minx) / m, (maxy - xy[:, 1]) / m])

plt.figure(figsize=(8, 9))
plt.imshow(IMG)
p = to_px(FP); plt.plot(*np.vstack([p, p[:1]]).T, color="#ff3b30", lw=2, label="GIS 외곽선 (바닥)")
plt.legend(loc="lower right", fontsize=12); plt.title(SRC, fontsize=12); plt.axis("off"); plt.show()

## 4. 기복변위 — 높은 건물의 지붕은 옆으로 밀려 찍힌다

위 그림에서 빨간 외곽선(**바닥**)과 실제 옥상이 어긋나 보인다. 위성·항공 영상은 바로 위에서만 찍지 않는다.
비스듬히 본 시선 때문에 **높은 곳일수록 더 멀리 밀려** 찍힌다 — 이것을 기복변위(relief displacement)라 한다.

$$\text{밀린 거리} \approx h \cdot \tan\theta \qquad (h = \text{높이},\ \theta = \text{연직에서 기운 촬영각})$$

94 m 건물을 15° 만 기울여 찍어도 25 m 가 밀린다. 바닥 외곽선으로 지붕을 자르면 **옆 건물의 지붕이나 도로**를 읽게 된다.
진짜 정사영상(true ortho)은 이걸 지형·건물 높이로 다 펴 놓은 영상인데, 무료 타일은 대개 그렇지 않다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
h, th = 1.0, math.radians(25)
ax.fill([0, 0.6, 0.6, 0], [0, 0, h, h], color="#c9d6e3", ec="#33475b", lw=2)
ax.plot([-0.4, 2.2], [0, 0], color="#555", lw=2)
for x in (0, 0.6):                       # 지붕 모서리에서 지면으로 내려오는 비스듬한 시선
    ax.annotate("", (x + h * math.tan(th), 0), (x, h),
                arrowprops=dict(arrowstyle="->", color="#ff9500", lw=1.8, ls="--"))
ax.plot([h * math.tan(th), 0.6 + h * math.tan(th)], [0, 0], color="#ff9500", lw=8, solid_capstyle="butt")
ax.plot([0, 0.6], [-0.03, -0.03], color="#ff3b30", lw=8, solid_capstyle="butt")
ax.text(0.3, -0.13, "바닥 외곽선 (GIS)", color="#ff3b30", ha="center", fontsize=13)
ax.text(0.3 + h * math.tan(th), 0.1, "영상에 찍힌 지붕", color="#ff9500", ha="center", fontsize=13)
ax.annotate("h", (-0.08, 0.5), fontsize=15, ha="center")
ax.text(1.55, 0.75, "비스듬한 시선\n(위성은 바로 위에 있지 않다)", fontsize=12, color="#ff9500")
ax.set_xlim(-0.4, 2.3); ax.set_ylim(-0.25, 1.15); ax.axis("off")
ax.set_title("기복변위: 지붕은 높이 × tan(촬영각) 만큼 밀려 찍힌다", fontsize=14); plt.show()

for deg in (5, 10, 15, 20, 24):
    print(f"촬영각 {deg:2d}° → 높이 {H:.0f} m 지붕이 {H * math.tan(math.radians(deg)):5.1f} m 밀린다")

## 5. 외곽선을 에지맵 위에서 밀어 보며 지붕을 찾는다

지붕은 **바닥과 모양이 같고 위치만 밀린** 다각형이다. 그러면

1. 영상의 **에지맵**(밝기가 급히 바뀌는 곳)을 만든다 — 지붕 테두리(파라펫)가 진하게 나온다.
2. 외곽선을 굵기 2 px 선으로 그린 **템플릿**을 만든다.
3. 템플릿을 사방 ±r 만큼 밀어 보며 에지와 얼마나 겹치는지 점수를 매긴다 (`cv2.matchTemplate`, 정규화 상관).
4. 점수가 비슷하면 **덜 민 쪽**을 고르도록 거리 벌점을 조금 준다 (`prior_weight`) — 옆 건물 테두리에 끌려가지 않게.

탐색 반경은 높이의 0.45배 (약 24° 촬영각까지). 원본 `estimate_roof_shift` 를 그대로 옮겼다.

In [ ]:
def max_relief_shift(height_m):
    """기복변위 상한: 촬영 경사각 ~24° 가정 → 높이의 0.45배 (등록 오차 여유 3 m)."""
    return float(max(3.0, 0.45 * height_m))

def estimate_roof_shift(img, bounds, mpp, footprint, max_shift_m, prior_weight=0.15):
    """footprint 외곽선 템플릿을 영상 에지맵에 정합 → 지붕 이동량 (dx 동, dy 북, m)."""
    minx, miny, maxx, maxy = bounds
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32), (0, 0), 1.0)
    mag = cv2.GaussianBlur(np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)), (0, 0), 1.0)
    px = to_px(footprint, bounds, mpp)
    x0, y0 = np.floor(px.min(0)).astype(int) - 3
    x1, y1 = np.ceil(px.max(0)).astype(int) + 3
    tpl = np.zeros((y1 - y0, x1 - x0), np.float32)
    cv2.polylines(tpl, [np.round(px - [x0, y0]).astype(np.int32)], True, 1.0, 2)
    r = int(max_shift_m / mpp)
    sx0, sy0 = max(0, x0 - r), max(0, y0 - r)                   # 탐색 영역: 원위치 ± r
    sx1, sy1 = min(mag.shape[1], x1 + r), min(mag.shape[0], y1 + r)
    region = mag[sy0:sy1, sx0:sx1]
    if region.shape[0] < tpl.shape[0] or region.shape[1] < tpl.shape[1]:
        return 0.0, 0.0, {"reason": "region_too_small"}
    score = cv2.matchTemplate(region, tpl, cv2.TM_CCORR_NORMED)
    oy, ox = np.mgrid[0:score.shape[0], 0:score.shape[1]]
    dxp, dyp = ox + sx0 - x0, oy + sy0 - y0                     # 원위치 대비 픽셀 이동
    dist = np.hypot(dxp, dyp) * mpp
    norm = (score - score.mean()) / (score.std() + 1e-6)
    final = norm - prior_weight * norm.std() * (dist / max(max_shift_m, 1.0)) * 4
    iy, ix = np.unravel_index(int(np.argmax(final)), final.shape)
    at0 = score[np.clip(y0 - sy0, 0, score.shape[0] - 1), np.clip(x0 - sx0, 0, score.shape[1] - 1)]
    info = {"score": float(score[iy, ix]), "score_at_zero": float(at0),
            "shift_px": [int(dxp[iy, ix]), int(dyp[iy, ix])],
            # 아래는 그림용
            "_mag": mag, "_tpl": tpl, "_final": final, "_extent": [dxp.min() * mpp, dxp.max() * mpp, -dyp.max() * mpp, -dyp.min() * mpp]}
    return float(dxp[iy, ix] * mpp), float(-dyp[iy, ix] * mpp), info

DX, DY, info = estimate_roof_shift(IMG, BOUNDS, MPP, FP, max_relief_shift(H))
shift = math.hypot(DX, DY)
print(f"지붕 이동량  동쪽 {DX:+.2f} m, 북쪽 {DY:+.2f} m  → {shift:.1f} m")
print(f"정합 점수    원위치 {info['score_at_zero']:.3f} → 보정 후 {info['score']:.3f}")
print(f"역산한 촬영각 ≈ atan({shift:.1f} / {H:.1f}) = {math.degrees(math.atan2(shift, H)):.1f}°")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6.5))
ax[0].imshow(info["_mag"], cmap="magma", vmax=np.percentile(info["_mag"], 99)); ax[0].set_title("에지맵 (Sobel 크기)", fontsize=14)
ax[1].imshow(info["_tpl"], cmap="gray"); ax[1].set_title("외곽선 템플릿 (2 px 선)", fontsize=14)
im = ax[2].imshow(info["_final"], cmap="viridis", extent=info["_extent"], origin="upper")
ax[2].plot(0, 0, "w+", ms=16, mew=2, label="원위치 (0, 0)")
ax[2].plot(DX, DY, "r*", ms=20, label=f"최고점 ({DX:+.1f}, {DY:+.1f}) m")
ax[2].set_xlabel("동쪽으로 민 거리 (m)"); ax[2].set_ylabel("북쪽으로 민 거리 (m)")
ax[2].legend(loc="lower right"); ax[2].set_title("밀어 본 위치별 점수 (거리 벌점 포함)", fontsize=14)
plt.colorbar(im, ax=ax[2], fraction=.046)
for a in ax[:2]: a.axis("off")
plt.tight_layout(); plt.show()

## 6. 보정 전 · 후

In [ ]:
FS = FP + np.array([DX, DY])                       # 지붕 외곽선 = 바닥 외곽선을 민 것
pad = 12
lo, hi = np.minimum(FP.min(0), FS.min(0)) - pad, np.maximum(FP.max(0), FS.max(0)) + pad
c0, r1 = to_px([[lo[0], lo[1]]])[0]; c1, r0 = to_px([[hi[0], hi[1]]])[0]
c0, r0, c1, r1 = int(max(c0, 0)), int(max(r0, 0)), int(c1), int(r1)
view = IMG[r0:r1, c0:c1]

fig, ax = plt.subplots(1, 2, figsize=(13, 9))
for a, poly, col, t in [(ax[0], FP, "#ff3b30", "보정 전 — 바닥 외곽선으로 자르면"),
                         (ax[1], FS, "#00c7be", f"보정 후 — {shift:.1f} m 옮긴 지붕 외곽선")]:
    a.imshow(view)
    p = to_px(poly) - [c0, r0]
    a.plot(*np.vstack([p, p[:1]]).T, color=col, lw=2.5)
    if a is ax[1]:
        q = to_px(FP) - [c0, r0]
        a.plot(*np.vstack([q, q[:1]]).T, color="#ff3b30", lw=1, ls="--", alpha=.8)
        a.annotate("", p.mean(0), q.mean(0), arrowprops=dict(arrowstyle="->", color="yellow", lw=2.5))
    a.set_title(t, fontsize=15); a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "roof_shift_overlay.png", dpi=110, bbox_inches="tight"); plt.show()

## 7. 지붕만 잘라낸다 — 건물 주축으로 세워서

옥상 설비(계단실·실외기)는 정북이 아니라 **건물이 앉은 방향**으로 놓여 있다.
비스듬한 건물을 정북 축 그대로 자르면, 판독 모델이 돌려주는 0~1 좌표가 건물이 아니라 지도 기준이 되어 설비가 어긋나 놓인다.
그래서 먼저 건물의 **주축 각도**만큼 영상을 되돌려 건물을 똑바로 세운 뒤 자르고, 그 각도를 함께 넘긴다.

주축 각도는 변 방향의 길이 가중 평균인데, 가로변과 세로변은 같은 축이므로 **각도를 4배 해서**(90° 주기) 평균을 낸다.
짧은 사선 변 하나에 끌려가지 않는다.

In [ ]:
def principal_angle_deg(ring):
    """외곽선의 주축 각도 (-45~45°, +X 기준 반시계)."""
    p = np.asarray(ring, float)
    e = np.roll(p, -1, axis=0) - p
    L = np.hypot(e[:, 0], e[:, 1])
    th = np.arctan2(e[:, 1], e[:, 0])
    z = (L * np.exp(4j * th)).sum()
    if abs(z) < 1e-9:
        return 0.0
    return float(((np.degrees(np.angle(z) / 4.0) + 45.0) % 90.0) - 45.0)

def aligned_roof_crop(img, bounds, mpp, fs):
    """보정된 지붕 외곽선 fs 로 자른 (마스킹본, 원본크롭, 가로 m, 세로 m, 주축 각도°, 크롭 안 외곽선 px)."""
    minx, miny, maxx, maxy = bounds
    axis = principal_angle_deg(fs)
    if abs(axis) < 0.75:                      # 거의 정북이면 돌리지 않는다 (재표본화로 화질만 잃는다)
        axis = 0.0
        bx0, by0 = fs.min(0); bx1, by1 = fs.max(0)
        c0, c1 = int((bx0 - minx) / mpp), int(math.ceil((bx1 - minx) / mpp))
        r0, r1 = int((maxy - by1) / mpp), int(math.ceil((maxy - by0) / mpp))
        crop = img[max(r0, 0):r1, max(c0, 0):c1].copy()
        local = np.column_stack([(fs[:, 0] - bx0) / mpp, (by1 - fs[:, 1]) / mpp])
    else:
        # 로컬(동/북) → 영상(가로/세로)은 y 부호가 뒤집혀서 화면상 회전 부호가 반대다
        cen = fs.mean(0)
        cpx = ((cen[0] - minx) / mpp, (maxy - cen[1]) / mpp)
        M = cv2.getRotationMatrix2D(cpx, -axis, 1.0)
        rot = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        rp = to_px(fs, bounds, mpp) @ M[:, :2].T + M[:, 2]
        c0, c1 = int(math.floor(rp[:, 0].min())), int(math.ceil(rp[:, 0].max()))
        r0, r1 = int(math.floor(rp[:, 1].min())), int(math.ceil(rp[:, 1].max()))
        crop = rot[max(r0, 0):r1, max(c0, 0):c1].copy()
        local = rp - np.array([max(c0, 0), max(r0, 0)])
    mask = np.zeros(crop.shape[:2], np.uint8)
    cv2.fillPoly(mask, [local.astype(np.int32)], 255)
    masked = crop.copy()
    masked[mask == 0] = 128                   # 외곽선 밖은 회색으로 지워 옥상만 남긴다
    return masked, crop, crop.shape[1] * mpp, crop.shape[0] * mpp, float(axis), local

ROOF, ROOF_RAW, RW, RD, AXIS, ROOF_POLY = aligned_roof_crop(IMG, BOUNDS, MPP, FS)
Image.fromarray(ROOF).save(OUT / "roof_crop.png")
Image.fromarray(ROOF_RAW).save(OUT / "roof_crop_raw.png")
print(f"주축 {AXIS:.1f}° 로 세워서 자름 → {ROOF.shape[1]}×{ROOF.shape[0]} px = {RW:.1f} × {RD:.1f} m")

real = np.array(Image.open(DATA / "roof_ortho.png").convert("RGB"))
fig, ax = plt.subplots(1, 3, figsize=(12, 8))
for a, im, t in [(ax[0], ROOF_RAW, "세워서 자른 원본"), (ax[1], ROOF, "외곽선 밖 지움 → 판독에 넣는 것"),
                 (ax[2], real, "실제 파이프라인이 만든 것\n(roof_ortho.png)")]:
    a.imshow(im); a.set_title(t, fontsize=13); a.axis("off")
plt.tight_layout(); plt.show()

## 8. 옥상 판독 — 비전 모델에게 **JSON 스키마**로 묻는다

판독 모델에게 "옥상에 뭐가 있어?" 라고 자유롭게 묻지 않는다. **답의 모양을 스키마로 고정**한다.

* `kind` — 정해진 15종 중 하나 (`enum`). 목록 밖 단어는 아예 나올 수 없다.
* `x_norm`, `y_norm` — 크롭 안 0~1 비율 위치. 미터가 아니라 비율이라 건물 크기와 무관하다.
* `w_m`, `d_m` — 크기. 크롭의 실제 폭·깊이(m)를 프롬프트로 알려 준다.
* `h_m` — **위에서는 높이를 알 수 없다.** 그림자로 추정될 때만 값, 아니면 `null` → 종류별 표준 높이를 쓴다.
* `strict: True` — OpenAI Responses API 가 스키마에 맞는 JSON 만 생성하게 강제한다.

모델 이름은 변수다. 파이프라인은 `gpt-5.5` 를 쓴다. 갖고 있는 다른 비전 모델로 바꿔 봐도 된다.

In [ ]:
KINDS = ["stair_core", "elevator_machine", "machine_room", "penthouse", "water_tank", "ac_cluster",
         "condenser", "chiller", "duct", "vent_pipe", "hatch", "drain", "antenna", "solar", "greenery"]

# 위에서는 높이를 알 수 없다 → 종류별 표준 높이(m)
STD_H = {"stair_core": 2.9, "elevator_machine": 2.6, "machine_room": 2.8, "penthouse": 2.9,
         "water_tank": 1.9, "ac_cluster": 0.9, "condenser": 1.2, "chiller": 1.6, "duct": 0.55,
         "vent_pipe": 1.3, "hatch": 0.25, "drain": 0.06, "antenna": 1.8, "solar": 0.5, "greenery": 0.4}

SCHEMA = {
    "type": "object", "additionalProperties": False,
    "required": ["items", "parapet_visible", "railing", "surface", "confidence", "notes"],
    "properties": {
        "items": {"type": "array", "items": {
            "type": "object", "additionalProperties": False,
            "required": ["kind", "x_norm", "y_norm", "w_m", "d_m", "h_m", "count", "confidence"],
            "properties": {
                "kind": {"type": "string", "enum": KINDS},
                "x_norm": {"type": "number"},          # 좌→우 0~1 (부재 중심)
                "y_norm": {"type": "number"},          # 하→상 0~1
                "w_m": {"type": "number"},
                "d_m": {"type": "number"},
                "h_m": {"anyOf": [{"type": "number"}, {"type": "null"}]},
                "count": {"type": "integer"},
                "confidence": {"type": "number"}}}},
        "parapet_visible": {"type": "boolean"},
        "railing": {"type": "string", "enum": ["none", "bars", "mesh"]},
        "surface": {"type": "string", "enum": ["waterproof_green", "waterproof_gray", "concrete", "tile", "gravel", "unknown"]},
        "confidence": {"type": "number"},
        "notes": {"type": "string"}}}

PROMPT = """You are reading a top-down orthophoto of ONE building's roof. The image is cropped to the
building's footprint bounding box: the full image width is {w_m:.1f} m and the height is {d_m:.1f} m.
The building has {floors} floors above ground and is about {height_m:.1f} m tall.

List the rooftop structures and equipment you can actually see. For each one give its CENTER as
x_norm (0 = left edge, 1 = right edge) and y_norm (0 = BOTTOM edge, 1 = top edge), and its footprint
size in metres (w_m across, d_m deep) using the scale above.

Rules:
- Only report what is visible. Do not invent typical equipment.
- A large bright box with a door is usually stair_core or penthouse; a round bright cylinder is water_tank.
- Repeated small boxes in a row are ac_cluster or condenser: report ONE item with count = how many.
- Long thin silver runs are duct. Small dark circles are drain. Thin vertical poles with a shadow are antenna.
- Dark blue/black rectangular grids are solar.
- h_m: only if the object's shadow length lets you estimate its height; otherwise null.
- confidence 0..1 per item.
Return at most 14 items, largest first."""
print(PROMPT.format(w_m=RW, d_m=RD, floors=len(geom["floors"]), height_m=H)[:400], "...")

In [ ]:
def to_data_url(img, max_side=1024, fmt="PNG"):
    """numpy 배열·PIL 이미지 → data URL (긴 변 max_side 이하로 줄여서)."""
    im = Image.fromarray(img) if isinstance(img, np.ndarray) else img.convert("RGB")
    s = max_side / max(im.size)
    if s < 1:
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, fmt, **({"quality": 90} if fmt == "JPEG" else {}))
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode()

ROOF_MODEL = "gpt-5.5"        # 파이프라인 기본값. 가진 비전 모델로 바꿔도 된다
REASONING = "low"             # 추론 모델이 아니면 None

raw, meta = None, {}
if OPENAI_KEY:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_KEY, timeout=180, max_retries=1)
        kwargs = dict(model=ROOF_MODEL,
                      input=[{"role": "user", "content": [
                          {"type": "input_text", "text": PROMPT.format(w_m=RW, d_m=RD, floors=len(geom["floors"]), height_m=H)},
                          {"type": "input_image", "image_url": to_data_url(ROOF, 1024)}]}],
                      text={"format": {"type": "json_schema", "name": "roof_reading", "schema": SCHEMA, "strict": True}})
        if REASONING:
            kwargs["reasoning"] = {"effort": REASONING}
        t0 = time.time()
        resp = client.responses.create(**kwargs)
        raw = json.loads(resp.output_text)
        u = resp.usage
        meta = {"source": f"openai:{ROOF_MODEL}", "latency_s": round(time.time() - t0, 1),
                "input_tokens": u.input_tokens, "output_tokens": u.output_tokens}
        print(f"{ROOF_MODEL} 판독 {meta['latency_s']}초 · 토큰 입력 {u.input_tokens} / 출력 {u.output_tokens}")
        print(json.dumps(raw, ensure_ascii=False, indent=1)[:1500])
    except Exception as e:
        print("판독 호출 실패 →", type(e).__name__, str(e)[:300])
        raw = None
if raw is None:
    print("API 판독 없이 진행 — 실제 파이프라인(gpt-5.5)이 이 건물 옥상에서 읽은 결과를 불러온다 (data/ipark/bds.json)")

### 판독을 3D 부재로 — 물리적으로 말이 되게 조인다

모델의 숫자를 그대로 믿지 않는다 (`_to_items` 이식).

* 신뢰도 0.35 미만은 버린다.
* 크기는 지붕의 90 % 를 넘지 못하고, 중심은 지붕 밖으로 나가지 못하게 조인다.
* 높이가 `null` 이면 종류별 표준 높이.
* "실외기 6대" 처럼 묶음으로 오면, 1대 크기로 환산하고 긴 쪽 방향으로 늘어놓는다.

In [ ]:
def to_items(raw_items, w_m, d_m):
    out = []
    for r in raw_items:
        if float(r.get("confidence", 0)) < 0.35 or r.get("kind") not in KINDS:
            continue
        kind = r["kind"]
        w = float(np.clip(r.get("w_m", 1.0), 0.2, w_m * 0.9))
        d = float(np.clip(r.get("d_m", 1.0), 0.2, d_m * 0.9))
        x = float(np.clip(r.get("x_norm", 0.5), w / w_m / 2, 1 - w / w_m / 2))
        y = float(np.clip(r.get("y_norm", 0.5), d / d_m / 2, 1 - d / d_m / 2))
        h = r.get("h_m")
        h = STD_H.get(kind, 1.0) if h is None else float(np.clip(h, 0.05, 12.0))
        cnt = max(1, min(12, int(r.get("count", 1))))
        axis = "x"
        if cnt > 1:                              # 묶음 → 1개 크기, 시작점을 왼쪽(아래) 끝으로
            axis = "x" if w >= d else "y"
            run, span = (w, w_m) if axis == "x" else (d, d_m)
            unit = max(0.35, (run - 0.4 * (cnt - 1)) / cnt)
            sh = (run - unit) / 2 / span
            if axis == "x":
                x, w = float(np.clip(x - sh, 0, 1)), unit
            else:
                y, d = float(np.clip(y - sh, 0, 1)), unit
        out.append({"kind": kind, "x_m": round(x, 4), "y_m": round(y, 4), "run_axis": axis,
                    "w_m": round(w, 2), "d_m": round(d, 2), "h_m": round(h, 2), "count": cnt,
                    "confidence": float(r.get("confidence", 0))})
    return out

if raw is not None:
    ITEMS = to_items(raw.get("items", []), RW, RD)
    ROOF_META = {**meta, "railing": raw.get("railing"), "surface": raw.get("surface"),
                 "confidence": raw.get("confidence"), "notes": raw.get("notes", "")}
else:
    bds = json.loads((DATA / "bds.json").read_text(encoding="utf-8"))
    ITEMS = [{k: it[k] for k in ("kind", "x_m", "y_m", "run_axis", "w_m", "d_m", "h_m", "count")}
             | {"confidence": it["ev"]["confidence"]} for it in bds["roof"]["items"]]
    ROOF_META = {"source": "staged:bds.json (실제 run, gpt-5.5)", "railing": bds["roof"]["railing"],
                 "axis_deg_real_run": bds["roof"]["axis_deg"]}

COL = {"stair_core": "#ff3b30", "penthouse": "#ff3b30", "elevator_machine": "#ff9500", "machine_room": "#ff9500",
       "water_tank": "#007aff", "ac_cluster": "#34c759", "condenser": "#34c759", "chiller": "#30b0c7",
       "duct": "#af52de", "solar": "#5856d6", "greenery": "#a2845e"}
from matplotlib.patches import Rectangle
fig, ax = plt.subplots(figsize=(6, 11))
ax.imshow(ROOF_RAW, extent=[0, RW, 0, RD])
for it in ITEMS:
    for k in range(it["count"]):
        step = (it["w_m"] if it["run_axis"] == "x" else it["d_m"]) + 0.4
        cx = it["x_m"] * RW + (k * step if it["run_axis"] == "x" else 0)
        cy = it["y_m"] * RD + (k * step if it["run_axis"] == "y" else 0)
        ax.add_patch(Rectangle((cx - it["w_m"] / 2, cy - it["d_m"] / 2), it["w_m"], it["d_m"],
                               fill=False, ec=COL.get(it["kind"], "white"), lw=2))
    ax.text(it["x_m"] * RW, it["y_m"] * RD, it["kind"], color="white", fontsize=10, ha="center",
            bbox=dict(fc=COL.get(it["kind"], "k"), alpha=.75, lw=0, pad=1))
ax.set_xlabel("m"); ax.set_title(f"옥상 부재 {len(ITEMS)}종\n({ROOF_META['source']})", fontsize=13); plt.show()

## 9. 넘기기 — `out/roof_items.json`

In [ ]:
roof_out = {
    "building": name, "frame": "LOCAL_ENU", "axis_deg": AXIS,
    "roof_shift_m": [round(DX, 2), round(DY, 2)], "crop_size_m": [round(RW, 2), round(RD, 2)],
    "ortho_source": SRC, "meta": ROOF_META, "items": ITEMS,
    "note": "x_m, y_m 은 주축(axis_deg)으로 세운 지붕 사각형 안의 0~1 비율. y 는 아래(0)→위(1).",
}
(OUT / "roof_items.json").write_text(json.dumps(roof_out, ensure_ascii=False, indent=1), encoding="utf-8")

print("── 요약 ─────────────────────────────")
print(f"정사영상   {SRC}")
print(f"기복변위   ({DX:+.2f}, {DY:+.2f}) m = {shift:.1f} m, 촬영각 ≈ {math.degrees(math.atan2(shift, H)):.1f}°")
print(f"지붕 크롭  {RW:.1f} × {RD:.1f} m, 주축 {AXIS:.1f}°")
print(f"옥상 판독  {ROOF_META['source']} → 부재 {len(ITEMS)}종 ({', '.join(sorted({i['kind'] for i in ITEMS}))})")
for f in ["ortho_local.png", "roof_shift_overlay.png", "roof_crop.png", "roof_crop_raw.png", "roof_items.json"]:
    print("  out/" + f, f"{(OUT / f).stat().st_size / 1024:.0f} KB")

## 해 볼 것

1. `ZOOM` 을 18 로 낮추면 기복변위 추정이 어떻게 달라지나? 픽셀이 커지면 몇 m 단위로만 움직일 수 있다.
2. `prior_weight` 를 0 으로 두고 다시 돌려 보자. 옆 건물 테두리에 끌려가는가? 벌점이 왜 필요한가.
3. 자기 동네 건물 하나로 해 보자 — 낮은 건물(5층)과 높은 건물(30층)의 이동량을 비교. 높이에 비례하는가?
4. `ROOF_MODEL` 을 다른 비전 모델로 바꿔 같은 크롭을 읽혀 보자. 부재 목록이 얼마나 다른가? 스키마가 무엇을 막아 주는가.
5. 위에서 본 영상으로는 **높이를 알 수 없다** — 그래서 표준 높이를 쓴다. 그림자 길이와 촬영 시각으로 높이를 추정하려면 무엇이 더 필요할까?

### 참고
* VWorld 오픈API — WMTS 위성영상 (`api.vworld.kr/req/wmts/1.0.0/{key}/Satellite/{z}/{y}/{x}.jpeg`)
* 기복변위(relief displacement)와 정사보정 — 사진측량 교재의 "항공사진의 기하" 장
* OpenAI Responses API — Structured Outputs (`text.format = json_schema, strict: true`)